In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path(
    "/Users/kesiyun/Desktop/5488 Big Data/5488Project Big Data/additional"
)

csv_files = sorted(data_dir.glob("*air_hourly.csv"))

all_df = []

for f in csv_files:

    print(f"\nReading {f.name}")

    try:
        # find header line
        with open(f, "r", encoding="utf-8", errors="ignore") as fp:
            lines = fp.readlines()

        header_idx = None

        for i, line in enumerate(lines):
            if line.strip().startswith("DATE,HOUR,STATION"):
                header_idx = i
                break

        if header_idx is None:
            print("  Cannot find CSV header. Skipped.")
            continue

        df = pd.read_csv(
            f,
            skiprows=header_idx,
            on_bad_lines="skip",
            na_values=["N.A.", "NA", "-", ""]
        )

        df["_source_file"] = f.name

        print(
            f"  rows={len(df):,} cols={len(df.columns)}"
        )

        all_df.append(df)

    except Exception as e:
        print(f"  FAILED: {e}")

raw = pd.concat(all_df, ignore_index=True)

print()
print("Combined shape:", raw.shape)

display(raw.head())


Reading 131230_180630air_hourly.csv
  rows=610,392 cols=11

Reading 180701_221231air_hourly.csv
  rows=675,120 cols=11

Reading 230101_260531air_hourly.csv
  rows=538,704 cols=11

Combined shape: (1824216, 11)


,DATE,HOUR,STATION,SO2,NOX,NO2,CO,RSP,O3,FSP,_source_file
0,30/12/2013,1,CAUSEWAY BAY,13.0,401.0,188.0,113.0,121.0,2.0,96.0,131230_180630air_hourly.csv
1,30/12/2013,2,CAUSEWAY BAY,11.0,251.0,142.0,116.0,115.0,9.0,91.0,131230_180630air_hourly.csv
2,30/12/2013,3,CAUSEWAY BAY,11.0,234.0,130.0,107.0,104.0,12.0,81.0,131230_180630air_hourly.csv
3,30/12/2013,4,CAUSEWAY BAY,10.0,188.0,115.0,107.0,94.0,13.0,73.0,131230_180630air_hourly.csv
4,30/12/2013,5,CAUSEWAY BAY,10.0,173.0,110.0,85.0,97.0,22.0,75.0,131230_180630air_hourly.csv


In [2]:
raw.columns = (
    raw.columns
    .str.strip()
    .str.upper()
)

print(raw.columns.tolist())

['DATE', 'HOUR', 'STATION', 'SO2', 'NOX', 'NO2', 'CO', 'RSP', 'O3', 'FSP', '_SOURCE_FILE']


In [3]:
raw["STATION"] = (
    raw["STATION"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Stations:", raw["STATION"].nunique())
print(sorted(raw["STATION"].dropna().unique()))

Stations: 18
['CAUSEWAY BAY', 'CENTRAL', 'CENTRAL/WESTERN', 'EASTERN', 'KWAI CHUNG', 'KWUN TONG', 'MONG KOK', 'NORTH', 'SHAM SHUI PO', 'SHATIN', 'SOUTHERN', 'TAI PO', 'TAP MUN', 'TSEUNG KWAN O', 'TSUEN WAN', 'TUEN MUN', 'TUNG CHUNG', 'YUEN LONG']


In [4]:
station_map = {
    s: s.title()
    for s in raw["STATION"].dropna().unique()
}

raw["station"] = raw["STATION"].map(station_map)

In [5]:
raw["HOUR"] = pd.to_numeric(
    raw["HOUR"],
    errors="coerce"
)

raw["hour"] = raw["HOUR"] - 1

raw["datetime"] = pd.to_datetime(
    raw["DATE"].astype(str)
    + " "
    + raw["hour"].astype("Int64").astype(str).str.zfill(2)
    + ":00",
    errors="coerce"
)

print(
    raw["datetime"].min(),
    "->",
    raw["datetime"].max()
)

print(
    "Failed datetime:",
    raw["datetime"].isna().sum()
)

/var/folders/dn/8xt3xqsn3ysf586z6l0zvfym0000gn/T/ipykernel_74532/246492399.py:8: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  raw["datetime"] = pd.to_datetime(


2013-12-30 00:00:00 -> 2026-05-31 23:00:00
Failed datetime: 0


In [6]:
pollutants = [
    "SO2",
    "NOX",
    "NO2",
    "CO",
    "RSP",
    "O3",
    "FSP"
]

for col in pollutants:
    raw[col] = pd.to_numeric(
        raw[col],
        errors="coerce"
    )

raw[pollutants].describe()

,SO2,NOX,NO2,CO,RSP,O3,FSP
count,1.774790e+06,1.657165e+06,1.772781e+06,1.031706e+06,1.764757e+06,1.774892e+06,1.762835e+06
mean,5.785530e+00,7.962206e+01,4.433883e+01,6.225757e+01,3.104136e+01,4.923721e+01,1.922280e+01
std,5.680231e+00,8.810549e+01,3.301433e+01,2.624430e+01,2.119366e+01,3.787841e+01,1.410039e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.000000e+00,2.500000e+01,2.100000e+01,4.400000e+01,1.600000e+01,1.900000e+01,9.000000e+00
50%,5.000000e+00,5.100000e+01,3.700000e+01,5.900000e+01,2.600000e+01,4.100000e+01,1.600000e+01
75%,7.000000e+00,9.800000e+01,5.900000e+01,7.800000e+01,4.100000e+01,7.200000e+01,2.500000e+01
max,3.010000e+02,1.835000e+03,4.960000e+02,3.590000e+02,4.020000e+02,4.340000e+02,2.960000e+02


In [7]:
print("Rows:", len(raw))

print(
    "Missing station:",
    raw["station"].isna().sum()
)

print(
    "Missing datetime:",
    raw["datetime"].isna().sum()
)

print(
    "Duplicate station-time:",
    raw.duplicated(
        subset=["station", "datetime"]
    ).sum()
)

Rows: 1824216
Missing station: 0
Missing datetime: 0
Duplicate station-time: 0


In [8]:
before = len(raw)

raw = raw.drop_duplicates(
    subset=["station", "datetime"],
    keep="first"
)

print(
    f"{before:,} -> {len(raw):,}"
)

1,824,216 -> 1,824,216


In [9]:
final = raw[
    [
        "datetime",
        "station",
        "SO2",
        "NOX",
        "NO2",
        "CO",
        "RSP",
        "O3",
        "FSP"
    ]
].copy()

final = final.sort_values(
    ["station", "datetime"]
).reset_index(drop=True)

print(final.shape)

display(final.head())

(1824216, 9)


,datetime,station,SO2,NOX,NO2,CO,RSP,O3,FSP
0,2013-12-30 00:00:00,Causeway Bay,13.0,401.0,188.0,113.0,121.0,2.0,96.0
1,2013-12-30 01:00:00,Causeway Bay,11.0,251.0,142.0,116.0,115.0,9.0,91.0
2,2013-12-30 02:00:00,Causeway Bay,11.0,234.0,130.0,107.0,104.0,12.0,81.0
3,2013-12-30 03:00:00,Causeway Bay,10.0,188.0,115.0,107.0,94.0,13.0,73.0
4,2013-12-30 04:00:00,Causeway Bay,10.0,173.0,110.0,85.0,97.0,22.0,75.0


In [10]:
roadside_stations = {
    "Central",
    "Mong Kok",
    "Causeway Bay"
}

final["station_type"] = np.where(
    final["station"].isin(roadside_stations),
    "roadside",
    "general"
)

final["is_roadside"] = (
    final["station_type"] == "roadside"
).astype(int)


In [11]:
final.to_parquet(
    "pollution_clean_long.parquet",
    index=False
)

final.to_csv(
    "pollution_clean_long.csv",
    index=False
)

print("Saved pollution_clean_long.parquet")
print(final.shape)

Saved pollution_clean_long.parquet
(1824216, 11)
